# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union (docs/SCHEMA_EXTENSION.md). `store.extend` extends the schemas a store holds;
`Builders.extend` extends the framework, the meta-schemas every store holds, once per process. Anything but added
members is refused. The cases that extend the framework come last: a notebook's cases share its process, and so the
framework's extensions.

In [ ]:
import { Builders as B, Paths, Plain, Validators } from "@mbse/schemas/Framework";
import { assert, entries_in, map, raises, same, spec_of } from "./support.js";

const store: any = new B.Store();
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const Int = store.Schemas.OfInt.as_schema(); // the store's own
const [Knows, Person, Customer, Id] = store.transaction(() => {
  const Knows = store.Schemas.OfRelation((r: any) => (
    r
      .name("crm.Knows")
      .links([(t: any) => t.name("a"), (t: any) => t.name("b")])
      .uniques([]) // a set
  ));
  const Person = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Person")
      .ref(true)
      .properties([
        (t: any) => t.name("name").schema([Text]),
        (t: any) => t.name("tags").schema([store.Schemas.OfContainer((c: any) => c.index([Int]).element([Text]))]),
      ])
      .adjacencies([(t: any) => t.name("knows").relation([Knows]).me("a")])
  ));
  const Customer = store.Schemas.OfObject((o: any) => (
    o
      .name("crm.Customer")
      .ref(true)
      .refines([Person])
      .properties([(t: any) => t.name("account").schema([Text])])
  ));
  store.Schemas.OfObject((o: any) => ( // a refinement of a refinement
    o
      .name("crm.Vip")
      .ref(true)
      .refines([Customer])
      .properties([(t: any) => t.name("tier").schema([Int])])
      .adjacencies([(t: any) => t.name("fans").relation([Knows]).me("b")])
  ));
  const Id = store.Schemas.OfUnion((u: any) => u.name("crm.Id").branches([(t: any) => t.name("n").schema([Int])]));
  return [Knows, Person, Customer, Id];
});
const EXTENDS = "an extension only adds members to";

## EXT-01 · `store.extend` adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
{
  const [ada] = store.transaction(() => {
    const ada = store.crm.Person((p: any) => p.name("Ada")); // built before the extension
    return [ada];
  });
  store.extend((m: any) => {
    m.Schemas.OfObject(Person, (o: any) => o.properties([(t: any) => t.name("age").schema([Int])]));
    m.Schemas.OfObject(Person, (o: any) => o.adjacencies([(t: any) => t.relation([Knows]).me("b")])); // anonymous
    m.Schemas.OfRelation(Knows, (r: any) => r.properties([(t: any) => t.name("since").schema([Int])]));
    m.Schemas.OfUnion(Id, (u: any) => u.branches([(t: any) => t.name("s").schema([Text])]));
    m.Schemas.OfObject((o: any) => o.name("crm.Note").properties([(t: any) => t.name("text").schema([Text])])); // built: its own
  });
  assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.anonymous.length === 1);
  assert(same([...Customer.properties.keys()], ["name", "tags", "age", "account"])); // inherited
  assert(same([...Knows.entry.properties.keys()], ["since"]) && same(Id.branches.map((b: any) => b.name), ["n", "s"]));
  store.transaction(() => {
    assert(ada.age.get() === null && store.crm.Person(ada, (p: any) => p.age(36n)).age.get() === 36n); // absent until written
    assert(store.crm.Customer((c: any) => c.name("Bo").age(7n).account("A")).age.get() === 7n);
  });
  assert(store.registered("crm.Note").validate().length === 0 && store.extend((m: any) => m) === store);
}

## EXT-02 · The extension check refuses anything but added members, saying what the extension would change

In [ ]:
for (const [extension, says] of [
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.description("A person")), "'crm.Person': it changes 'description'"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.ref(false)), "'crm.Person': it changes 'ref'"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.properties((adv: any) => adv.remove((e: any) => e.name("name")))),
    "'crm.Person': it removes or replaces what 'properties' holds"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.properties([(t: any) => t.name("name").schema([Int])])), // retyped
    "'crm.Person': it removes or replaces what 'properties' holds"],
  [(m: any) => m.Schemas.OfRelation(Knows, (r: any) => r.links([(t: any) => t.name("c")])),
    "'crm.Knows': it adds to 'links', which an extension cannot"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.properties([(t: any) => t.name("account").schema([Text])])),
    "'crm.Person': it adds 'account', which 'crm.Customer' declares"],
  [(m: any) => m.Schemas.OfObject(Customer, (o: any) => o.properties([(t: any) => t.name("name").schema([Text])])),
    "'crm.Customer': it adds 'name', which it inherits"],
  [(m: any) => m.Schemas.OfObject(Customer, (o: any) => o.adjacencies([(t: any) => t.name("name").relation([Knows]).me("a")])),
    "'crm.Customer': it adds 'name', which it inherits"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.adjacencies([(t: any) => t.name("name").relation([Knows]).me("a")])),
    "'crm.Person': it adds 'name', which it has"],
  [(m: any) => m.Schemas.OfObject(Customer, (o: any) => o.adjacencies([(t: any) => t.name("knows").relation([Knows]).me("a")])),
    "'crm.Customer': it adds 'knows', which it inherits"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.properties([(t: any) => t.name("tier").schema([Int])])),
    "'crm.Person': it adds 'tier', which 'crm.Vip' declares"],
  [(m: any) => m.Schemas.OfObject(Person, (o: any) => o.properties([(t: any) => t.name("fans").schema([Int])])),
    "'crm.Person': it adds 'fans', which 'crm.Vip' declares"],
  [(m: any) => m.Schemas.OfContainer(Person.properties.get("tags").schema, (c: any) => c.layout("sparse")), // unnamed: its kind
    "a container schema: it changes 'layout'"],
] as const) {
  raises(TypeError, () => store.extend(extension), `${EXTENDS} ${says}`);
}
assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.ref && Person.description === null); // refused: unwritten
store.transaction(() => {
  store.Schemas.OfObject(Person, (o: any) => o.description("A person")); // outside an extension, a store's own schema changes
});
assert(Person.description === "A person");

## EXT-03 · A framework schema, which every store holds, changes only through `Builders.extend`

In [ ]:
store.transaction(() => {
  for (const change of [(o: any) => o.description("x"), (o: any) => o.properties([(t: any) => t.name("unit").schema([Text])])]) {
    raises(TypeError, () => store.Schemas.OfObject(store.Schemas.OfScalar, change), // outside an extension: neither changed nor added to
      "'Schemas.OfScalar' is one of the schemas the store was made holding: extend it with store.extend");
  }
});
raises(TypeError, () => store.extend((m: any) => m.Schemas.OfObject(m.Schemas.OfScalar, (o: any) => o.description("x"))),
  `${EXTENDS} 'Schemas.OfScalar': it changes 'description'`);
assert(!store.Schemas.OfScalar.as_schema().properties.has("unit") && store.Schemas.OfScalar.as_schema().description === null);

## EXT-04 · `Builders.extend` extends the framework once per process: every store holds what it built, and a meta-schema's new members are its schemas' builders'

In [ ]:
/** Domains, a scalar's domain and a quantity's unit: what mbse-expressions adds to a store, which extends its own
 * `Schemas.OfScalar` and `Schemas.OfObject`. */
function domains(m: any): void {
  const Domain = m.Schemas.OfObject((o: any) => (
    o
      .name("Expressions.Framework.Domain")
      .ref(true)
      .properties([(t: any) => t.name("label").schema([m.Schemas.OfStr])])
  ));
  const HasDomain = m.Schemas.OfRelation((r: any) => (
    r
      .name("Expressions.Framework.HasDomain")
      .links([(t: any) => t.name("schema"), (t: any) => t.name("domain")])
      .uniques("schema")
  ));
  m.Schemas.OfObject(Domain, (o: any) => o.adjacencies([(t: any) => t.name("schemas").relation([HasDomain]).me("domain")]));
  m.Schemas.OfObject(m.Schemas.OfScalar, (o: any) => o.adjacencies([
    (t: any) => t.name("domain").relation([HasDomain]).me("schema"),
  ]));
  m.Schemas.OfObject(m.Schemas.OfObject, (o: any) => o.properties([(t: any) => t.name("unit").schema([m.Schemas.OfStr])]));
}

const [extended, plain]: any[] = [new B.Store().extend(domains), new B.Store()];
assert(extended.Schemas.OfScalar.as_schema().adjacencies.has("domain") && extended.Schemas.OfObject.as_schema().properties.has("unit"));
assert(!plain.Schemas.OfScalar.as_schema().adjacencies.has("domain") && !plain.names().includes("Expressions.Framework.Domain"));
assert(!extended.framework(extended.registered("Expressions.Framework.Domain"))); // what it builds is the store's own
const [Meters] = extended.transaction(() => {
  const Meters = extended.Schemas.OfObject((o: any) => (
    o
      .name("Meters")
      .properties([(t: any) => t.name("value").schema([extended.Schemas.OfFloat])])
      .unit("m")
  ));
  return [Meters];
});
assert(new B.OfObject.Builder(Meters, false, extended).unit.get() === "m");
const [Feet] = extended.transaction(() => {
  const Feet = new B.OfObject.Builder(Meters, true, extended).name("Feet").clone();
  return [Feet];
});
assert(new B.OfObject.Builder(Feet, false, extended).unit.get() === "m"); // a clone keeps it
assert(new B.OfObject.Builder(plain.Schemas.OfObject, false, plain).unit.get() === null); // another store's builders lack it
assert(new B.OfScalar.Builder(plain.Schemas.OfInt, false, plain).domain.get() === null);
raises(TypeError, () => (new B.Store() as any).extend((m: any) => m.Schemas.OfObject(m.Schemas.OfScalar, (o: any) => o.adjacencies([
  (t: any) => t.name("trio").relation([m.Schemas.OfRelation((r: any) => r.name("T3").links([
    (u: any) => u.name("a"), (u: any) => u.name("b"), (u: any) => u.name("c")]))]).me("a")]))),
"an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'");

## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot; what a store lacking the extension ignores

In [ ]:
{
  const s: any = new B.Store().extend(domains);
  const [int8, Byte] = s.transaction(() => {
    const int8 = s.Expressions.Framework.Domain((d: any) => d.label("int8"));
    const Byte = s.Schemas.OfScalar(s.Schemas.OfInt, (x: any) => x.domain([int8])); // from the schema's end: the store's own scalar
    return [int8, Byte];
  });
  assert(Byte === s.Schemas.OfInt.as_schema() && same([...int8.schemas].map((e: any) => e.schema === Byte), [true])); // seen from the data's end
  const [int16] = s.transaction(() => {
    const int16 = s.Expressions.Framework.Domain((d: any) => d.label("int16").schemas((a: any) => a.insert((e: any) => e.schema(Byte))));
    return [int16];
  });
  assert(same([...new B.OfScalar.Builder(Byte, false, s).domain].map((e: any) => e.domain.label.get()), ["int8", "int16"]));
  assert(Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8).length === 0);
  const text = (value: unknown) => JSON.stringify(value, (_k, v) => (v instanceof Map ? Object.fromEntries(v) : typeof v === "bigint" ? `${v}n` : v));
  const log = Plain.ToPlain(s).Snapshot() as any[];
  const held = log[1].get("mutations")[0].get("upsert").get("exists")[0].get("symbols"); // the data, and the scalar its entries link
  const written = [...held.keys()].flatMap((symbol: string) => ["schemas", "domain"].flatMap((name) => entries_in(spec_of(log, symbol, 1), name)));
  assert(written.length === 2 && written.every((e) => e.has("schema"))); // each once, from the data's end
  const back: any = new B.Store().extend(domains); // a round trip between stores with the same extensions
  Plain.FromPlain(back).Snapshot(log);
  assert([...new B.OfScalar.Builder(back.Schemas.OfInt, false, back).domain].length === 2);
  assert(text(Plain.ToPlain(back).Snapshot()) === text(log));
  s.transaction(() => {
    s.Schemas.OfScalar(Byte, (x: any) => x.domain((a: any) => a.remove((e: any) => e.domain(int8))));
  });
  assert([...int8.schemas].length === 0 && [...int16.schemas].length === 1);
  // A store's basic scalar is its own: the entries linking it are the store's, and another store's is never linked.
  const [one, other]: any[] = [new B.Store().extend(domains), new B.Store().extend(domains)];
  one.transaction(() => {
    one.Expressions.Framework.Domain((d: any) => d.label("machine int").schemas((a: any) => a.insert((e: any) => e.schema(one.Schemas.OfFloat))));
  });
  const seen = (store: any) => [...new B.OfScalar.Builder(store.Schemas.OfFloat, false, store).domain].map((e: any) => e.domain.label.get());
  assert(same(seen(one), ["machine int"]) && same(seen(other), []));
  const third: any = new B.Store().extend(domains);
  Plain.FromPlain(third).Snapshot(Plain.ToPlain(one).Snapshot());
  assert(same(seen(third), ["machine int"]) && same(seen(other), []));
  assert(text(Plain.ToPlain(third).Snapshot()) === text(Plain.ToPlain(one).Snapshot())); // read back as written
  one.transaction(() => {
    raises(TypeError, () => one.Expressions.Framework.Domain((d: any) => d.schemas((a: any) => a.insert((e: any) => e.schema(
      other.Schemas.OfInt)))), "proxies can only link proxy instances, or schema objects their store holds");
  });
  // Read into a store without an extension, a sub-clause whose meaning needs it is ignored, and kept where asked: a
  // property a schema the store was made holding lacks, an entry only such a schema could hold, a property of an entry of
  // such a relation. The rest is read.
  const bare: any = new B.Store();
  let ignored: Plain.Ignored[] = [];
  Plain.FromPlain(bare).Snapshot(Plain.ToPlain(extended).Snapshot(), ignored);
  assert(same(ignored.map((i) => i.reason), ["Schemas.OfObject has no property 'unit'", "Schemas.OfObject has no property 'unit'"])); // Meters', Feet's
  assert(ignored.every((i) => i.path.endsWith(".as_property")) && same([...bare.registered("Meters").properties.keys()], ["value"]));
  /** A tag a scalar has, declared from the scalar's end only. */
  const tagged = (m: any): void => {
    const Tags = m.Schemas.OfRelation((r: any) => r.name("X.Tags").links([(t: any) => t.name("scalar"), (t: any) => t.name("tag")]));
    m.Schemas.OfObject((o: any) => o.name("X.Tag").ref(true));
    m.Schemas.OfObject(m.Schemas.OfScalar, (o: any) => o.adjacencies([(t: any) => t.name("tags").relation([Tags]).me("scalar")]));
  };
  const labels: any = new B.Store().extend(tagged);
  labels.transaction(() => {
    labels.Schemas.OfScalar(labels.Schemas.OfInt, (x: any) => x.tags([labels.X.Tag((x: any) => x)]));
    labels.Schemas.OfObject((o: any) => o.name("Note").properties([(t: any) => t.name("text").schema([labels.Schemas.OfStr])]));
  });
  ignored = [];
  Plain.FromPlain(new B.Store()).Snapshot(Plain.ToPlain(labels).Snapshot(), ignored);
  assert(same(ignored.map((i) => i.reason), ["Schemas.OfScalar has no adjacency 'tags'"])); // written from the scalar's end
  const altered = Plain.ToPlain(labels).Snapshot() as any[]; // as an extension adding a property to Schemas.HasSchema would write it
  const entry = spec_of(altered, "Note").find((m) => m.has("as_adjacency")).get("as_adjacency").get("mutations")[0].get("insert").get("entries")[0];
  const typing = entry.get("mutations").find((p: any) => p.has("as_link")).get("as_link").get("mutations")[0].get("set").get("def").get("mutations");
  const typed = typing.find((m: any) => m.has("as_adjacency") && m.get("as_adjacency").get("name") === "schema");
  typed.get("as_adjacency").get("mutations")[0].get("insert").get("entries")[0].get("mutations").push(
    map({ as_property: { name: "why", mutations: [{ set: { value: "sized" } }] } }));
  ignored = [];
  Plain.FromPlain(new B.Store()).Snapshot(altered, ignored);
  assert(same(ignored.map((i) => i.reason), [
    "an entry of 'schema' has no property 'why'", "Schemas.OfScalar has no adjacency 'tags'"]));
  assert(Plain.FromPlain(new B.Store()).Snapshot(altered).length > 0); // ignored, kept or not

  /** A scalar's shapes, object schemas, declared from the scalar's end only. */
  const shaped = (m: any): void => {
    const Shapes = m.Schemas.OfRelation((r: any) => r.name("X.Shapes").links([(t: any) => t.name("scalar"), (t: any) => t.name("shape")]));
    m.Schemas.OfObject(m.Schemas.OfScalar, (o: any) => o.adjacencies([(t: any) => t.name("shapes").relation([Shapes]).me("scalar")]));
  };
  // An entry written from a store's own scalar, which a snapshot never writes, is written still, and what it links that
  // has no name of its own follows it.
  const shapes: any = new B.Store().extend(shaped);
  shapes.transaction(() => {
    const shape = shapes.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("x").schema([shapes.Schemas.OfInt])])); // unnamed
    shapes.Schemas.OfScalar(shapes.Schemas.OfInt, (x: any) => x.shapes([shape]));
  });
  const shaping = Plain.ToPlain(shapes).Snapshot();
  const again: any = new B.Store().extend(shaped);
  Plain.FromPlain(again).Snapshot(shaping);
  assert([...new B.OfScalar.Builder(again.Schemas.OfInt, false, again).shapes].length === 1
    && text(Plain.ToPlain(again).Snapshot()) === text(shaping));
  assert(Plain.FromPlain(again).Snapshot(shaping).length === 0 && [...new B.OfScalar.Builder(again.Schemas.OfInt, false, again).shapes].length === 1);

  /** A board, a singleton, pinning schemas: data that links schema objects. */
  const pinned = (m: any): void => {
    const Pins = m.Schemas.OfRelation((r: any) => r.name("X.Pins").links([(t: any) => t.name("board"), (t: any) => t.name("schema")]));
    m.Schemas.OfObject((o: any) => o.name("X.Board").ref(true).singleton("x.Board").adjacencies([
      (t: any) => t.name("pins").relation([Pins]).me("board")]));
  };
  // Read again, a snapshot adds nothing: what an entry defines where it is linked is made where the object it is written
  // with held no such entry before (a singleton, which every store holding its schema has, or a store's own scalar).
  const boards: any = new B.Store().extend(pinned);
  boards.transaction(() => {
    const unnamed = [boards.Schemas.OfStr, boards.Schemas.OfInt].map((element: any) =>
      boards.Schemas.OfContainer((c: any) => c.index([boards.Schemas.OfInt]).element([element])));
    boards.X.Board(boards.singleton("x.Board"), (b: any) => b.pins((a: any) => a.insert((e: any) => e.schema(unnamed[0])).insert(
      (e: any) => e.schema(unnamed[1]))));
  });
  const pinning = Plain.ToPlain(boards).Snapshot();
  const pinnedAgain: any = new B.Store().extend(pinned);
  const board = pinnedAgain.singleton("x.Board");
  assert(same(Plain.FromPlain(pinnedAgain).Snapshot(pinning).map((made) => made === board), [true])); // its data, the board
  assert(same(Plain.FromPlain(pinnedAgain).Snapshot(pinning).map((made) => made === board), [true])); // read again: found
  assert([...board.pins].length === 2);
  assert(same([...board.pins].map((e: any) => e.schema.element),
    [pinnedAgain.Schemas.OfStr.as_schema(), pinnedAgain.Schemas.OfInt.as_schema()]));

  /** Tags an object schema has, declared from its end: any object schema's, the store's own included. */
  const objectsTagged = (m: any): void => {
    const Tags = m.Schemas.OfRelation((r: any) => r.name("X.Tags").links([(t: any) => t.name("object"), (t: any) => t.name("tag")]));
    m.Schemas.OfObject((o: any) => o.name("X.Tag").ref(true));
    m.Schemas.OfObject(m.Schemas.OfObject, (o: any) => o.adjacencies([(t: any) => t.name("tags").relation([Tags]).me("object")]));
  };
  // The store's own object schema, holding entries of the store's own relations, is written with its extension's alone.
  const own: any = new B.Store().extend(objectsTagged);
  own.transaction(() => {
    own.Schemas.OfObject(own.Schemas.OfObject, (x: any) => x.tags([own.X.Tag((t: any) => t)]));
  });
  const owned = Plain.ToPlain(own).Snapshot() as any[];
  assert(same(spec_of(owned, "Schemas.OfObject", 1).filter((m) => m.has("as_adjacency")).map((m) => m.get("as_adjacency").get("name")), ["tags"]));
  const mirror: any = new B.Store().extend(objectsTagged);
  Plain.FromPlain(mirror).Snapshot(owned);
  assert([...new B.OfObject.Builder(mirror.Schemas.OfObject, false, mirror).tags].length === 1
    && text(Plain.ToPlain(mirror).Snapshot()) === text(owned));
}